## Het feit rit S2

In [1]:
#SETUP
import ConnectionConfig as cc
cc.setupEnvironment()
from pyspark.sql import SparkSession

In [2]:
#SETUP
spark = cc.startLocalCluster("factRidesSaif")
spark.getActiveSession()


In [3]:
#EXTRACT
cc.set_connectionProfile("VeloDB")

# ride_source_df = spark.read \
#     .format("jdbc") \
#     .option("url", cc.create_jdbc()) \
#     .option("driver", cc.get_Property("driver")) \
#     .option("dbtable", "rides") \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideId") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0) \
#     .option("upperBound", 1000) \
#     .load()

ride_source_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("query", """
        SELECT *,
            haversine_km(
                CAST(r.startpoint[0] AS NUMERIC), 
                CAST(r.startpoint[1] AS NUMERIC),
                CAST(r.endpoint[0] AS NUMERIC), 
                CAST(r.endpoint[1] AS NUMERIC)
            ) AS distance
        FROM rides r
    """) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

In [4]:
ride_source_df.createOrReplaceTempView("rides_source_saif")

In [5]:

# spark.sql("SELECT * FROM rides_source_saif").show(5)

In [6]:
#EXTRACT
dim_customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLocks")

In [7]:
#TRANSFORM
dim_customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")

In [8]:
ride_source_df.createOrReplaceTempView("rides_source")

In [9]:
#TRANSFORM
fact_rides = spark.sql("""
    SELECT 
        r.rideId AS rideId,
        l1.lockSK AS startLockSK,
        l2.lockSK AS endLockSK,
        c.customer_SK AS customerSK,
        r.distance  as rideDistance
        
    FROM
        rides_source AS r
    LEFT OUTER JOIN 
        dimLocks AS l1 ON r.startlockId = l1.lockId
    LEFT OUTER JOIN 
        dimLocks AS l2 ON r.endlockId = l2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON r.subscriptionid = c.subscription_id
    WHERE 
        l1.lockId IS NOT NULL AND l2.lockId IS NOT NULL AND c.customer_id IS NOT NULL
        and r.starttime >= c.scd_start and (r.starttime < c.scd_end  OR c.scd_end is null)
""")

In [10]:
#LOAD
fact_rides.show(20)
# fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRidesS2")

+------+-----------+---------+----------+--------------------+
|rideId|startLockSK|endLockSK|customerSK|        rideDistance|
+------+-----------+---------+----------+--------------------+
|173306|       5246|     1426|8589992417|2.502671075950720000|
|173312|       5389|     1003|8589960690|2.584487953569830000|
|173313|       7315|     3673|8589983021|6.371203055060470000|
|173316|       6306|     7315|8589936889|3.747045354813460000|
|173318|       2385|     4079|8589956739|2.226097604431930000|
|173319|       5787|     6098|8589973924|3.376639973621110000|
|173323|       1286|     1837|8589945832|6.512013059229540000|
|173324|       5671|     6114|8589969300|1.797206734648380000|
|173327|       2447|      199|8589994239|0.854163260048668000|
|173328|       5583|     5884|8589992790|1.154797672210640000|
|173329|       6098|     5997|8589976621|1.112890081231000000|
|173330|       3944|     4070|8589943621|1.936483571558220000|
|173331|       1837|     6555|8589945944|1.617916266570

In [11]:
delta_table_path = "spark-warehouse/factRidesS2"
fact_rides.write.format("delta").mode("overwrite").save(delta_table_path)

In [12]:
spark.stop()